<a href="https://colab.research.google.com/github/Josh012006/OpenX-Embodiment-Datasets-Visualization/blob/main/colabs/Open_X_Embodiment_Datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Copyright 2020 DeepMind Technologies Limited.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# Open X-Embodiment Datasets

![](https://robotics-transformer-x.github.io/img/overview.png)



This colab helps you **visualize** the end-effector positions and task descriptions across datasets in the Open X-Embodiment Dataset, for the training episodes. It is based on the initial one here :

https://github.com/google-deepmind/open_x_embodiment/blob/main/colabs/Open_X_Embodiment_Datasets.ipynb

# Individual dataset visualization
![](https://raw.githubusercontent.com/Josh012006/OpenX-Embodiment-Datasets-Visualization/main/public/individual.png)

# Combined dataset visualisation
![](https://raw.githubusercontent.com/Josh012006/OpenX-Embodiment-Datasets-Visualization/main/public/combined.png)


# Datasets infos

In [ ]:
# Install dependencies
%pip install "tensorflow-metadata<1.21.0" tensorflow tensorflow-datasets matplotlib numpy gcsfs ipywidgets plotly

In [ ]:
import numpy as np
import plotly.graph_objects as go
import tensorflow_datasets as tfds
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

DATASETS = [
    'fractal20220817_data',
    'kuka',
    'bridge',
    'taco_play',
    'jaco_play',
    'berkeley_cable_routing',
    'roboturk',
    'nyu_door_opening_surprising_effectiveness',
    'viola',
    'berkeley_autolab_ur5',
    'toto',
    'language_table',
    'columbia_cairlab_pusht_real',
    'stanford_kuka_multimodal_dataset_converted_externally_to_rlds',
    'nyu_rot_dataset_converted_externally_to_rlds',
    'stanford_hydra_dataset_converted_externally_to_rlds',
    'austin_buds_dataset_converted_externally_to_rlds',
    'nyu_franka_play_dataset_converted_externally_to_rlds',
    'maniskill_dataset_converted_externally_to_rlds',
    'cmu_franka_exploration_dataset_converted_externally_to_rlds',
    'ucsd_kitchen_dataset_converted_externally_to_rlds',
    'ucsd_pick_and_place_dataset_converted_externally_to_rlds',
    'austin_sailor_dataset_converted_externally_to_rlds',
    'austin_sirius_dataset_converted_externally_to_rlds',
    'bc_z',
    'usc_cloth_sim_converted_externally_to_rlds',
    'utokyo_pr2_opening_fridge_converted_externally_to_rlds',
    'utokyo_pr2_tabletop_manipulation_converted_externally_to_rlds',
    'utokyo_saytap_converted_externally_to_rlds',
    'utokyo_xarm_pick_and_place_converted_externally_to_rlds',
    'utokyo_xarm_bimanual_converted_externally_to_rlds',
    'robo_net',
    'berkeley_mvp_converted_externally_to_rlds',
    'berkeley_rpt_converted_externally_to_rlds',
    'kaist_nonprehensile_converted_externally_to_rlds',
    'stanford_mask_vit_converted_externally_to_rlds',
    'tokyo_u_lsmo_converted_externally_to_rlds',
    'dlr_sara_pour_converted_externally_to_rlds',
    'dlr_sara_grid_clamp_converted_externally_to_rlds',
    'dlr_edan_shared_control_converted_externally_to_rlds',
    'asu_table_top_converted_externally_to_rlds',
    'stanford_robocook_converted_externally_to_rlds',
    'eth_agent_affordances',
    'imperialcollege_sawyer_wrist_cam',
    'iamlab_cmu_pickup_insert_converted_externally_to_rlds',
    'uiuc_d3field',
    'utaustin_mutex',
    'berkeley_fanuc_manipulation',
    'cmu_play_fusion',
    'cmu_stretch',
    'berkeley_gnm_recon',
    'berkeley_gnm_cory_hall',
    'berkeley_gnm_sac_son'
]

In [ ]:
# NOTE: Robot/gripper info below is best-effort, compiled from the OXE paper,
# TFDS dataset descriptions, and related publications. A handful of entries
# (e.g. eth_agent_affordances, imperialcollege_sawyer_wrist_cam, the dlr_*
# datasets) are sparsely documented and were not individually re-verified.
# If precision matters for a specific dataset, check its TFDS catalog page
# (https://www.tensorflow.org/datasets/catalog/<dataset_name>) or its
# original paper before relying on this for analysis.

DATASET_ROBOT_INFO = {
    "fractal20220817_data":         {"robot": "Google Robot",        "gripper": "2-finger"},
    "kuka":                         {"robot": "KUKA iiwa",           "gripper": "2-finger"},
    "bridge":                       {"robot": "WidowX",              "gripper": "2-finger"},
    "taco_play":                    {"robot": "Franka",              "gripper": "2-finger"},
    "jaco_play":                    {"robot": "Jaco 2",              "gripper": "3-finger"},
    "berkeley_cable_routing":       {"robot": "Franka",              "gripper": "2-finger"},
    "roboturk":                     {"robot": "Sawyer",              "gripper": "2-finger"},
    "nyu_door_opening_surprising_effectiveness": {"robot": "Hello Stretch", "gripper": "2-finger"},
    "viola":                        {"robot": "Franka",              "gripper": "2-finger"},
    "berkeley_autolab_ur5":         {"robot": "UR5",                 "gripper": "Robotiq 2F-85"},
    "toto":                         {"robot": "Franka",              "gripper": "2-finger"},
    "language_table":               {"robot": "xArm",                "gripper": "cylindrical pusher"},
    "columbia_cairlab_pusht_real":  {"robot": "UR5",                 "gripper": "cylindrical pusher"},
    "stanford_kuka_multimodal_dataset_converted_externally_to_rlds": {"robot": "KUKA iiwa", "gripper": "2-finger"},
    "nyu_rot_dataset_converted_externally_to_rlds": {"robot": "xArm", "gripper": "2-finger"},
    "stanford_hydra_dataset_converted_externally_to_rlds": {"robot": "Franka", "gripper": "2-finger"},
    "austin_buds_dataset_converted_externally_to_rlds": {"robot": "Franka", "gripper": "2-finger"},
    "nyu_franka_play_dataset_converted_externally_to_rlds": {"robot": "Franka", "gripper": "2-finger"},
    "maniskill_dataset_converted_externally_to_rlds": {"robot": "Franka", "gripper": "2-finger"},
    "cmu_franka_exploration_dataset_converted_externally_to_rlds": {"robot": "Franka", "gripper": "2-finger"},
    "ucsd_kitchen_dataset_converted_externally_to_rlds": {"robot": "xArm", "gripper": "2-finger"},
    "ucsd_pick_and_place_dataset_converted_externally_to_rlds": {"robot": "xArm", "gripper": "2-finger"},
    "austin_sailor_dataset_converted_externally_to_rlds": {"robot": "Franka", "gripper": "2-finger"},
    "austin_sirius_dataset_converted_externally_to_rlds": {"robot": "Franka", "gripper": "2-finger"},
    "bc_z":                         {"robot": "Google Robot",        "gripper": "2-finger"},
    "usc_cloth_sim_converted_externally_to_rlds": {"robot": "Franka", "gripper": "2-finger"},
    "utokyo_pr2_opening_fridge_converted_externally_to_rlds": {"robot": "PR2", "gripper": "2-finger"},
    "utokyo_pr2_tabletop_manipulation_converted_externally_to_rlds": {"robot": "PR2", "gripper": "2-finger"},
    "utokyo_saytap_converted_externally_to_rlds": {"robot": "Unitree A1", "gripper": "N/A (quadruped)"},
    "utokyo_xarm_pick_and_place_converted_externally_to_rlds": {"robot": "xArm", "gripper": "2-finger"},
    "utokyo_xarm_bimanual_converted_externally_to_rlds": {"robot": "xArm (bimanual)", "gripper": "2-finger"},
    "robo_net":                     {"robot": "Multi-Robot",         "gripper": "various"},
    "berkeley_mvp_converted_externally_to_rlds": {"robot": "xArm",  "gripper": "2-finger"},
    "berkeley_rpt_converted_externally_to_rlds": {"robot": "Franka", "gripper": "2-finger"},
    "kaist_nonprehensile_converted_externally_to_rlds": {"robot": "Franka", "gripper": "2-finger"},
    "stanford_mask_vit_converted_externally_to_rlds": {"robot": "Sawyer", "gripper": "2-finger"},
    "tokyo_u_lsmo_converted_externally_to_rlds": {"robot": "Cobotta", "gripper": "2-finger"},
    "dlr_sara_pour_converted_externally_to_rlds": {"robot": "DLR SARA", "gripper": "Robotiq 2F-85"},
    "dlr_sara_grid_clamp_converted_externally_to_rlds": {"robot": "DLR SARA", "gripper": "Robotiq 2F-140"},
    "dlr_edan_shared_control_converted_externally_to_rlds": {"robot": "DLR EDAN", "gripper": "CLASH hand"},
    "asu_table_top_converted_externally_to_rlds": {"robot": "UR5",   "gripper": "Robotiq 2F-85"},
    "stanford_robocook_converted_externally_to_rlds": {"robot": "Franka", "gripper": "2-finger"},
    "eth_agent_affordances":        {"robot": "Franka",              "gripper": "2-finger"},
    "imperialcollege_sawyer_wrist_cam": {"robot": "Sawyer",          "gripper": "2-finger"},
    "iamlab_cmu_pickup_insert_converted_externally_to_rlds": {"robot": "Franka", "gripper": "2-finger"},
    "uiuc_d3field":                 {"robot": "Kinova Gen3",         "gripper": "Robotiq 2F-85"},
    "utaustin_mutex":               {"robot": "Franka",              "gripper": "2-finger"},
    "berkeley_fanuc_manipulation":  {"robot": "Fanuc Mate",          "gripper": "2-finger"},
    "cmu_play_fusion":              {"robot": "Franka",              "gripper": "2-finger"},
    "cmu_stretch":                  {"robot": "Hello Stretch",       "gripper": "2-finger"},
    "berkeley_gnm_recon":           {"robot": "Jackal",              "gripper": "N/A (navigation)"},
    "berkeley_gnm_cory_hall":       {"robot": "RC Car",              "gripper": "N/A (navigation)"},
    "berkeley_gnm_sac_son":         {"robot": "TurtleBot2",          "gripper": "N/A (navigation)"},
}

In [ ]:
# Utils
def dataset2path(dataset_name):
  if dataset_name == 'robo_net':
    version = '1.0.0'
  elif dataset_name == 'language_table':
    version = '0.0.1'
  else:
    version = '0.1.0'
  return f'gs://gresearch/robotics/{dataset_name}/{version}'

def extract_endpoint(step, config):
    """Retrieves the x, y, z positions of the end effector at an episode step."""
    data = step
    for key in config["field"]:
        data = data[key]
    data = data.numpy()

    if config["indices"] is not None:
        data = data[config["indices"]]

    if config["reshape"]:
      flat = data.flatten()
      if len(flat) >= 16:
          matrix = flat[:16].reshape(4, 4)
          convention = config.get("reshape_convention", "row")
          if convention == "col":
              return matrix[:3, 3]
          else:
              return matrix[3, :3]
      else:
          raise ValueError(f"Cannot reshape data of size {len(flat)} into 4x4 matrix")
    else:
        return data

def get_safe_split(b, max_episodes=500):
    """Returns the number of episodes to take (used with shuffle+take)."""
    try:
        total = b.info.splits['train'].num_examples
        return min(max_episodes, total)
    except Exception:
        return max_episodes

def normalize(endpoints):
    """Normalizes the endpoints to values between 0 and 1 for each axis."""
    mins = endpoints.min(axis=0)
    maxs = endpoints.max(axis=0)
    ranges = maxs - mins
    ranges[ranges == 0] = 1
    return (endpoints - mins) / ranges

## End effector coordinates fields

Visualizing the available features and record the end effector fields for each dataset where it is available.

In [ ]:
for dataset_name in DATASETS:
    print(f"\n{'='*60}")
    print(f"DATASET: {dataset_name}")
    print('='*60)
    try:
        b = tfds.builder_from_directory(builder_dir=dataset2path(dataset_name))
        print(b.info.features)
    except Exception as e:
        print(f"ERROR: {e}")

In [ ]:
DATASET_EEF_CONFIG = {
    "fractal20220817_data": {
        "field": ["observation", "base_pose_tool_reached"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "kuka": {
        "field": ["observation", "clip_function_input/base_pose_tool_reached"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "jaco_play": {
        "field": ["observation", "end_effector_cartesian_pos"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "taco_play": {
        "field": ["observation", "robot_obs"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "maniskill_dataset_converted_externally_to_rlds": {
        "field": ["observation", "tcp_pose"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "stanford_kuka_multimodal_dataset_converted_externally_to_rlds": {
        "field": ["observation", "ee_position"],
        "indices": None,
        "reshape": False
    },
    "nyu_rot_dataset_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "stanford_hydra_dataset_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "nyu_franka_play_dataset_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(7, 10),
        "reshape": False
    },
    "austin_sailor_dataset_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "bc_z": {
        "field": ["observation", "present/xyz"],
        "indices": None,
        "reshape": False
    },
    "utokyo_pr2_opening_fridge_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "utokyo_pr2_tabletop_manipulation_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "utokyo_xarm_pick_and_place_converted_externally_to_rlds": {
        "field": ["observation", "end_effector_pose"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "robo_net": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "berkeley_mvp_converted_externally_to_rlds": {
        "field": ["observation", "pose"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "stanford_mask_vit_converted_externally_to_rlds": {
        "field": ["observation", "end_effector_pose"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "tokyo_u_lsmo_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "dlr_sara_pour_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "dlr_sara_grid_clamp_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "dlr_edan_shared_control_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "eth_agent_affordances": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "stanford_robocook_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "berkeley_fanuc_manipulation": {
        "field": ["observation", "end_effector_state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "asu_table_top_converted_externally_to_rlds": {
        "field": ["ground_truth_states", "EE"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "kaist_nonprehensile_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(14, 17),
        "reshape": False
    },
    "ucsd_pick_and_place_dataset_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(0, 3),
        "reshape": False
    },
    "iamlab_cmu_pickup_insert_converted_externally_to_rlds": {
        "field": ["action"],
        "indices": slice(0, 3),
        "reshape": False
    },
    # --- Reshape required ---
    "viola": {
        "field": ["observation", "ee_states"],
        "indices": None,
        "reshape": True,
    },
    "austin_buds_dataset_converted_externally_to_rlds": {
        "field": ["observation", "state"],
        "indices": slice(8, 24),
        "reshape": True,
    },
    "austin_sirius_dataset_converted_externally_to_rlds": {
        "field": ["observation", "state_ee"],
        "indices": None,
        "reshape": True,
    },
    "utaustin_mutex": {
        "field": ["observation", "state"],
        "indices": slice(8, 24),
        "reshape": True,
    },
    "uiuc_d3field": {
        "field": ["observation", "state"],
        "indices": None,
        "reshape": True,
        "reshape_convention": "col"
    },
}

The datasets present in `DATASETS` but excluded from `DATASET_EEF_CONFIG` and `DATASET_ROBOT_INFO` were excluded for the following reasons:

**No usable EEF position field:**
- `bridge` — `state` shape (7,) contains joint states, not EEF position
- `berkeley_cable_routing` — `robot_state` shape (7,) not clearly documented
- `roboturk` — no state at all in the observation
- `nyu_door_opening_surprising_effectiveness` — no state
- `berkeley_autolab_ur5` — `robot_state` shape (15,) not documented
- `toto` — joint angles only
- `language_table` — 2D (XY) only, no Z
- `columbia_cairlab_pusht_real` — `robot_state` shape (2,) XY only
- `cmu_franka_exploration_dataset_converted_externally_to_rlds` — no EEF state
- `ucsd_kitchen_dataset_converted_externally_to_rlds` — joint states only
- `usc_cloth_sim_converted_externally_to_rlds` — no robot state
- `imperialcollege_sawyer_wrist_cam` — `state` shape (1,) gripper only
- `cmu_play_fusion` — joint angles only
- `cmu_stretch` — ambiguous
- `berkeley_rpt_converted_externally_to_rlds` — joint positions only

**Navigation, not manipulation:**
- `berkeley_gnm_recon`, `berkeley_gnm_cory_hall`, `berkeley_gnm_sac_son` — 2D navigation robots

**Non-manipulator robot:**
- `utokyo_saytap_converted_externally_to_rlds` — quadruped robot

**Bimanual removed:**
- `utokyo_xarm_bimanual_converted_externally_to_rlds` — two arms, ambiguous which one to choose

## Visualize the end effector final position

For all the datasets where it is available and for all the available episodes.

### Download cached info from github

In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output
import subprocess
import numpy as np
import os
import gc
import tensorflow as tf
import tensorflow_datasets as tfds

In [ ]:
REPO_URL = "https://github.com/Josh012006/OpenX-Embodiment-Datasets-Visualization.git"
REPO_DIR = "/content/OpenX-Embodiment-Datasets-Visualization"
CACHE_DIR = f"{REPO_DIR}/endpoints_cache_random"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)
    print("✅ Repo cloned")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print("✅ Repo updated")

available = [f.replace(".npy", "") for f in os.listdir(CACHE_DIR) if f.endswith(".npy")]
print(f"✅ {len(available)} datasets available:")
for name in sorted(available):
    print(f"  - {name}")

In [ ]:
def load_dataset(dataset_name, seed=42, shuffle_buffer=50):
    cache_path = f"{CACHE_DIR}/{dataset_name}.npy"

    if os.path.exists(cache_path):
        print(f"  ✅ {dataset_name}: already on disk")
        return

    config = DATASET_EEF_CONFIG[dataset_name]
    try:
        b = tfds.builder_from_directory(builder_dir=dataset2path(dataset_name))
        n_episodes = get_safe_split(b, max_episodes=500)
        read_config = tfds.ReadConfig(
            interleave_cycle_length=1,
            interleave_block_length=1,
        )
        # Random sampling instead of sequential slicing:
        # shuffle_files mixes shard read order, .shuffle() mixes elements
        # within a buffer, .take() picks n from that mixed stream.
        ds = b.as_dataset(split='train', read_config=read_config, shuffle_files=True)
        ds = ds.shuffle(buffer_size=shuffle_buffer, seed=seed)
        ds = ds.take(n_episodes)

        endpoints = []
        for episode in ds:
            last_step = None
            for step in episode["steps"]:
                last_step = step
            if last_step is not None:
                try:
                    xyz = extract_endpoint(last_step, config)
                    endpoints.append(xyz)
                except Exception:
                    pass
            del last_step
            gc.collect()

        if endpoints:
            np.save(cache_path, np.array(endpoints))
            print(f"  ✅ {dataset_name}: {len(endpoints)} episodes saved")
        else:
            print(f"  ❌ {dataset_name}: no valid endpoints")

    except Exception as e:
        print(f"  ❌ {dataset_name}: {e}")
    finally:
        gc.collect()
        tf.keras.backend.clear_session()

# Generate only missing .npy
missing = [name for name in DATASET_EEF_CONFIG
           if not os.path.exists(f"{CACHE_DIR}/{name}.npy")]

if missing:
    print(f"{len(missing)} datasets missing, generating...")
    for name in missing:
        load_dataset(name)
    print("Done!")
else:
    print("✅ All datasets already on disk.")

In [ ]:
OPENVLA_DATASETS = {
    "fractal20220817_data", "kuka", "taco_play", "jaco_play", "viola",
    "stanford_hydra_dataset_converted_externally_to_rlds",
    "austin_buds_dataset_converted_externally_to_rlds",
    "nyu_franka_play_dataset_converted_externally_to_rlds",
    "austin_sailor_dataset_converted_externally_to_rlds",
    "austin_sirius_dataset_converted_externally_to_rlds",
    "dlr_edan_shared_control_converted_externally_to_rlds",
    "iamlab_cmu_pickup_insert_converted_externally_to_rlds",
    "utaustin_mutex", "berkeley_fanuc_manipulation", "bc_z"
}

CACHE_DIR = "/content/OpenX-Embodiment-Datasets-Visualization/endpoints_cache_random"

AXIS_MAP = {'XY': (0, 1), 'XZ': (0, 2), 'YZ': (1, 2)}
AXIS_LABELS = {0: 'X', 1: 'Y', 2: 'Z'}
COLORS = [
    '#e6194b','#3cb44b','#4363d8','#f58231','#911eb4',
    '#0891b2','#f032e6','#65a30d','#e11d48','#469990',
    '#7c3aed','#9A6324','#ca8a04','#800000','#059669',
    '#808000','#c2410c','#000075','#a9a9a9','#6b7280',
    '#374151','#6d28d9','#b45309','#4169e1','#8B0000',
    '#00CED1','#be185d','#32CD32','#FF8C00','#9400D3',
    '#047857','#FF4500','#1E90FF',
]

def get_endpoints(dataset_name):
    cache_path = f"{CACHE_DIR}/{dataset_name}.npy"
    if os.path.exists(cache_path):
        return np.load(cache_path)
    return None

def normalize(endpoints):
    mins = endpoints.min(axis=0)
    maxs = endpoints.max(axis=0)
    ranges = maxs - mins
    ranges[ranges == 0] = 1
    return (endpoints - mins) / ranges

def get_dataset_label(dataset_name):
    info = DATASET_ROBOT_INFO.get(dataset_name, {})
    robot = info.get("robot", "Unknown")
    tag = "✅ OpenVLA" if dataset_name in OPENVLA_DATASETS else "🔵 OOD"
    return robot, tag

def make_base_marker_3d(scale=0.1):
    traces = []
    # Axes colorés
    for vec, color, label in [
        ([scale, 0, 0], 'red',   'X axis'),
        ([0, scale, 0], 'green', 'Y axis'),
        ([0, 0, scale], 'blue',  'Z axis')
    ]:
        traces.append(go.Scatter3d(
            x=[0, vec[0]], y=[0, vec[1]], z=[0, vec[2]],
            mode='lines',
            line=dict(color=color, width=6),
            name=label, showlegend=False
        ))
    # Point central
    traces.append(go.Scatter3d(
        x=[0], y=[0], z=[0],
        mode='markers+text',
        marker=dict(size=5, color='black'),
        text=['Base'], textposition='top center',
        name='Robot Base (0,0,0)', showlegend=True
    ))
    return traces

def make_base_marker_2d():
    return go.Scatter(
        x=[0], y=[0], mode='markers+text',
        marker=dict(size=10, color='black', symbol='cross'),
        text=['Base'], textposition='top center',
        name='Robot Base (0,0,0)'
    )

def make_trace_3d(endpoints, name, color):
    return go.Scatter3d(
        x=endpoints[:, 0], y=endpoints[:, 1], z=endpoints[:, 2],
        mode='markers',
        marker=dict(size=3, opacity=0.6, color=color),
        name=name
    )

def make_trace_2d(endpoints, name, color, axis1, axis2):
    return go.Scatter(
        x=endpoints[:, axis1], y=endpoints[:, axis2],
        mode='markers',
        marker=dict(size=4, opacity=0.6, color=color),
        name=name
    )

def build_figure(endpoints, dataset_name, view, normalize_flag, color='#4363d8'):
    is_3d = (view == '3D')
    robot, tag = get_dataset_label(dataset_name)
    n_episodes = len(np.load(f"{CACHE_DIR}/{dataset_name}.npy"))
    title = f"{dataset_name} | {robot} | {tag} | n={n_episodes}"
    suffix = ' (normalized)' if normalize_flag else ' (m)'

    fig = go.Figure()
    if is_3d:
        for trace in make_base_marker_3d():
            fig.add_trace(trace)
        fig.add_trace(make_trace_3d(endpoints, dataset_name, color))
        fig.update_layout(
            title=title,
            scene=dict(
                xaxis_title='X' + suffix,
                yaxis_title='Y' + suffix,
                zaxis_title='Z' + suffix,
                aspectmode='cube'
            ),
            width=1000, height=800
        )
    else:
        a1, a2 = AXIS_MAP[view]
        fig.add_trace(make_base_marker_2d())
        fig.add_trace(make_trace_2d(endpoints, dataset_name, color, a1, a2))
        fig.update_layout(
            title=title,
            xaxis_title=AXIS_LABELS[a1] + suffix,
            yaxis_title=AXIS_LABELS[a2] + suffix,
            width=1000, height=700
        )
    return fig

### Individual visualization


In [ ]:
# Choose the dataset to visualize
DATASET_NAME = "fractal20220817_data"  # <-- change here

# --- Widgets ---
ind_normalize_cb = widgets.Checkbox(value=False, description='Normalize')
ind_view_radio = widgets.RadioButtons(
    options=['3D', 'XY', 'XZ', 'YZ'],
    value='3D',
    description='View:',
    layout=widgets.Layout(width='250px')
)
ind_output = widgets.Output()

def ind_on_change(change):
    with ind_output:
        clear_output(wait=True)
        endpoints = get_endpoints(DATASET_NAME)
        if endpoints is None:
            print(f"❌ {DATASET_NAME} not found in cache")
            return
        if ind_normalize_cb.value:
            endpoints = normalize(endpoints)
        n = len(endpoints)
        fig = build_figure(endpoints, DATASET_NAME, ind_view_radio.value, ind_normalize_cb.value)
        fig.update_layout(title=fig.layout.title.text + f" | n={n}")
        display(fig)

ind_normalize_cb.observe(ind_on_change, names='value')
ind_view_radio.observe(ind_on_change, names='value')

controls_row = widgets.HBox([
    ind_normalize_cb,
    ind_view_radio,
])

display(widgets.VBox([controls_row, ind_output]))
ind_on_change(None)

### Combined visualization

In [ ]:
dataset_names = list(DATASET_EEF_CONFIG.keys())
checkboxes = {
    name: widgets.Checkbox(
        value=(i < 3),  # True for the first 3
        description=name,
        layout=widgets.Layout(width='350px')
    )
    for i, name in enumerate(dataset_names)
}

normalize_cb = widgets.Checkbox(value=False, description='Normalize per dataset')
view_radio = widgets.RadioButtons(
    options=['3D', 'XY', 'XZ', 'YZ'],
    value='3D',
    description='View:',
    layout=widgets.Layout(width='150px')
)

output = widgets.Output()
color_map = {name: COLORS[i % len(COLORS)] for i, name in enumerate(DATASET_EEF_CONFIG)}

def on_change(change):
    with output:
        clear_output(wait=True)
        selected = [name for name, cb in checkboxes.items() if cb.value]
        if not selected:
            print("No datasets selected.")
            return

        should_normalize = normalize_cb.value
        view = view_radio.value
        is_3d = (view == '3D')

        fig = go.Figure()
        if is_3d:
            for trace in make_base_marker_3d():
                fig.add_trace(trace)
        else:
            fig.add_trace(make_base_marker_2d())

        for dataset_name in selected:
            endpoints = get_endpoints(dataset_name)
            if endpoints is None:
                print(f"  ⚠️ {dataset_name}: not on disk")
                continue
            if should_normalize:
                endpoints = normalize(endpoints)

            robot, tag = get_dataset_label(dataset_name)
            n_episodes = len(endpoints)
            label = f"{dataset_name}<br>{robot} | {tag} | n={n_episodes}"
            color = color_map[dataset_name]

            if is_3d:
                fig.add_trace(make_trace_3d(endpoints, label, color))
            else:
                a1, a2 = AXIS_MAP[view]
                fig.add_trace(make_trace_2d(endpoints, label, color, a1, a2))

        suffix = ' (normalized)' if should_normalize else ' (m)'
        if is_3d:
            fig.update_layout(
                title="EEF Endpoint Distribution — Combined 3D",
                scene=dict(
                    xaxis_title='X' + suffix,
                    yaxis_title='Y' + suffix,
                    zaxis_title='Z' + suffix,
                    aspectmode='cube'
                ),
                width=1000, height=800,
                legend=dict(
                    x=1.05,
                    y=0.5,
                    itemsizing='constant',
                    itemwidth=60
                )
            )
        else:
            a1, a2 = AXIS_MAP[view]
            fig.update_layout(
                title=f"EEF Endpoint Distribution — Combined {view}",
                xaxis_title=AXIS_LABELS[a1] + suffix,
                yaxis_title=AXIS_LABELS[a2] + suffix,
                width=1000, height=700,
                legend=dict(
                    x=1.05,
                    y=0.5,
                    itemsizing='constant',
                    itemwidth=60
                )
            )
        display(fig)

for cb in checkboxes.values():
    cb.observe(on_change, names='value')
normalize_cb.observe(on_change, names='value')
view_radio.observe(on_change, names='value')

controls = widgets.VBox([
    widgets.HTML("<b>Options</b>"),
    normalize_cb,
    widgets.HTML("<hr>"),
    view_radio,
    widgets.HTML("<hr>"),
    widgets.HTML("<b>Datasets</b>"),
    widgets.VBox(list(checkboxes.values()))
])

on_change(None)
display(widgets.HBox([controls, output]))

## Task description embeddings

This section visualizes task descriptions across OXE datasets in a semantic space, using sentence embeddings + UMAP dimensionality reduction. Unlike the EEF endpoint visualization, this captures differences in *what the task is*, not just *where the arm moved* — addressing the coordinate-frame ambiguity issue raised when comparing spatial distributions across datasets with different reference frames.

In [ ]:
# Install dependencies for task description embeddings
%pip install sentence-transformers umap-learn

### Extract unique task descriptions

In [ ]:
# Inspect each dataset's features to confirm where the language instruction lives.
# Run this once before generating the cache, to catch datasets that need an
# override in TASK_FIELD_CONFIG (or that have no language instruction at all).
for dataset_name in DATASETS:
    print(f"\n{'='*60}")
    print(f"DATASET: {dataset_name}")
    print('='*60)
    try:
        b = tfds.builder_from_directory(builder_dir=dataset2path(dataset_name))
        steps_features = b.info.features['steps']
        if 'language_instruction' in steps_features:
            print("✅ has 'language_instruction' directly under steps")
        else:
            print("⚠️  no 'language_instruction' field found under steps — inspect manually:")
            print(steps_features)
    except Exception as e:
        print(f"❌ ERROR: {e}")

In [ ]:
# Field where the task description / language instruction is stored, per dataset.
# Most OXE datasets use "language_instruction" directly under steps.
# Datasets not listed here will fall back to the default field name below.
DEFAULT_INSTRUCTION_FIELD = "language_instruction"

TASK_FIELD_CONFIG = {
    "fractal20220817_data": {"field": ["observation", "natural_language_instruction"]},
    "kuka": {"field": ["observation", "natural_language_instruction"]},
    "bridge": {"field": ["observation", "natural_language_instruction"]},
    "taco_play": {"field": ["observation", "natural_language_instruction"]},
    "jaco_play": {"field": ["observation", "natural_language_instruction"]},
    "berkeley_cable_routing": {"field": ["observation", "natural_language_instruction"]},
    "roboturk": {"field": ["observation", "natural_language_instruction"]},
    "nyu_door_opening_surprising_effectiveness": {"field": ["observation", "natural_language_instruction"]},
    "viola": {"field": ["observation", "natural_language_instruction"]},
    "berkeley_autolab_ur5": {"field": ["observation", "natural_language_instruction"]},
    "toto": {"field": ["observation", "natural_language_instruction"]},
    "columbia_cairlab_pusht_real": {"field": ["observation", "natural_language_instruction"]},
    "bc_z": {"field": ["observation", "natural_language_instruction"]},
    # language_table: instruction is encoded as int32 bytes, not a plain
    # string — skipped for now, would need separate decoding logic.
}

# Datasets to skip entirely for this section
TASK_SKIP_DATASETS = {
    "language_table",   # instruction encoded as int32, not a plain string
    "bridge",           # incomplete cache — OOM during extraction
    "robo_net",         # incomplete cache — OOM during extraction
    "uiuc_d3field",     # language_instruction field present but always empty
}

def get_instruction_field(dataset_name):
    """Returns the list of nested keys to reach the instruction field."""
    if dataset_name in TASK_FIELD_CONFIG:
        return TASK_FIELD_CONFIG[dataset_name]["field"]
    return [DEFAULT_INSTRUCTION_FIELD]

The datasets present in `DATASETS` but excluded from this task description section was excluded for the following reason:


**Incompatible instruction encoding:**

- ``anguage_table` — instruction is encoded as a Tensor(shape=(512,), dtype=int32) of UTF-8 bytes, not a plain string field, requiring separate decoding logic not implemented here

**Incomplete cache — OOM during cluster extraction:**
- `bridge` — ~25,460 episodes with 480×640 images; the extraction job ran out of memory before completing the full dataset scan
- `robo_net` — ~162,000 episodes; same OOM issue, subprocess killed before any descriptions were saved

**No usable task descriptions:**
- `uiuc_d3field` — the `language_instruction` field exists in the TFDS schema but contains only empty strings across all episodes in the `train` split

### Download cached info from github

In [ ]:
TASK_REPO_DIR = "/content/OpenX-Embodiment-Datasets-Visualization"
TASK_CACHE_DIR = f"{TASK_REPO_DIR}/task_description_cache"

if not os.path.exists(TASK_REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)
    print("✅ Repo cloned")
else:
    subprocess.run(["git", "-C", TASK_REPO_DIR, "pull"], check=True)
    print("✅ Repo updated")

task_available = [f.replace(".json", "") for f in os.listdir(TASK_CACHE_DIR) if f.endswith(".json")]
print(f"✅ {len(task_available)} datasets with cached task descriptions:")
for name in sorted(task_available):
    print(f"  - {name}")

In [ ]:
import json

def extract_unique_task_descriptions(dataset_name):
    cache_path = f"{TASK_CACHE_DIR}/{dataset_name}.json"

    if os.path.exists(cache_path):
        print(f"  ✅ {dataset_name}: already on disk")
        return

    field_path = get_instruction_field(dataset_name)
    try:
        b = tfds.builder_from_directory(builder_dir=dataset2path(dataset_name))
        read_config = tfds.ReadConfig(
            interleave_cycle_length=1,
            interleave_block_length=1,
        )
        # No shuffle/take here: we want full coverage to catch rare task
        # descriptions, not a sample. Sets are small (text), so this is cheap.
        ds = b.as_dataset(split='train', read_config=read_config)

        unique_descriptions = set()
        for episode in ds:
            for step in episode["steps"]:
                try:
                    data = step
                    for key in field_path:
                        data = data[key]
                    instr = data.numpy()
                    if isinstance(instr, bytes):
                        instr = instr.decode('utf-8')
                    instr = str(instr).strip()
                    if instr:
                        unique_descriptions.add(instr)
                except Exception:
                    pass
            gc.collect()

        if unique_descriptions:
            with open(cache_path, 'w') as f:
                json.dump(sorted(unique_descriptions), f, indent=2)
            print(f"  ✅ {dataset_name}: {len(unique_descriptions)} unique task descriptions saved")
        else:
            print(f"  ❌ {dataset_name}: no valid task descriptions found")

    except Exception as e:
        print(f"  ❌ {dataset_name}: {e}")
    finally:
        gc.collect()
        tf.keras.backend.clear_session()

# Generate only missing .json files (skip incompatible datasets)
task_missing = [name for name in DATASETS
                 if name not in TASK_SKIP_DATASETS
                 and not os.path.exists(f"{TASK_CACHE_DIR}/{name}.json")]

if task_missing:
    print(f"{len(task_missing)} datasets missing, generating...")
    for name in task_missing:
        extract_unique_task_descriptions(name)
    print("Done!")
else:
    print("✅ All datasets already on disk.")

In [ ]:
from sentence_transformers import SentenceTransformer
import umap

# Load the embedding model once
EMBED_MODEL = SentenceTransformer('all-mpnet-base-v2')

# Cache for the 3D UMAP projection, separate from the raw description cache
UMAP_CACHE_PATH = f"{TASK_REPO_DIR}/task_description_cache/_umap_3d_cache.json"

def get_task_descriptions(dataset_name):
    cache_path = f"{TASK_CACHE_DIR}/{dataset_name}.json"
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            return json.load(f)
    return None

def compute_all_embeddings_3d(force_recompute=False, n_neighbors=30, min_dist=0.05, seed=42):
    """
    Computes a single shared UMAP projection across ALL cached datasets at once.
    This is important: UMAP must be fit on the full combined set of descriptions
    so that distances between datasets are meaningful. Fitting UMAP separately
    per dataset would put every dataset in its own incomparable coordinate space.
    """
    if os.path.exists(UMAP_CACHE_PATH) and not force_recompute:
        with open(UMAP_CACHE_PATH) as f:
            return json.load(f)

    # Gather all unique descriptions across all cached datasets
    all_descriptions = []
    all_dataset_labels = []
    for fname in sorted(os.listdir(TASK_CACHE_DIR)):
        if not fname.endswith(".json") or fname.startswith("_"):
            continue
        dataset_name = fname.replace(".json", "")
        descriptions = get_task_descriptions(dataset_name)
        if not descriptions:
            continue
        all_descriptions.extend(descriptions)
        all_dataset_labels.extend([dataset_name] * len(descriptions))

    print(f"Embedding {len(all_descriptions)} unique task descriptions across {len(set(all_dataset_labels))} datasets...")
    embeddings = EMBED_MODEL.encode(all_descriptions, show_progress_bar=True)

    print("Running UMAP (3D)...")
    reducer = umap.UMAP(
        n_components=3,
        n_neighbors=n_neighbors,   # higher = more global structure preserved
        min_dist=min_dist,         # lower = tighter, more accurate local clusters
        random_state=seed,
    )
    coords_3d = reducer.fit_transform(embeddings)

    result = {
        "dataset": all_dataset_labels,
        "description": all_descriptions,
        "x": coords_3d[:, 0].tolist(),
        "y": coords_3d[:, 1].tolist(),
        "z": coords_3d[:, 2].tolist(),
    }
    with open(UMAP_CACHE_PATH, 'w') as f:
        json.dump(result, f)
    print(f"✅ UMAP projection cached to {UMAP_CACHE_PATH}")
    return result

def get_dataset_points_3d(dataset_name, umap_data):
    """Filters the shared UMAP projection down to a single dataset's points."""
    idx = [i for i, d in enumerate(umap_data["dataset"]) if d == dataset_name]
    if not idx:
        return None
    return {
        "x": [umap_data["x"][i] for i in idx],
        "y": [umap_data["y"][i] for i in idx],
        "z": [umap_data["z"][i] for i in idx],
        "description": [umap_data["description"][i] for i in idx],
    }

def make_task_trace_3d(points, name, color):
    return go.Scatter3d(
        x=points["x"], y=points["y"], z=points["z"],
        mode='markers',
        marker=dict(size=4, opacity=0.7, color=color),
        text=points["description"],
        hovertemplate="%{text}<extra>" + name + "</extra>",
        name=name
    )

def build_task_figure(points, dataset_name, color='#4363d8'):
    robot, tag = get_dataset_label(dataset_name)
    n = len(points["x"])
    title = f"{dataset_name} | {robot} | {tag} | n={n} unique tasks"

    fig = go.Figure()
    fig.add_trace(make_task_trace_3d(points, dataset_name, color))
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title='UMAP-1',
            yaxis_title='UMAP-2',
            zaxis_title='UMAP-3',
            aspectmode='cube'
        ),
        width=1000, height=800
    )
    return fig

### Individual visualization

In [ ]:
# Compute (or load cached) shared UMAP projection across all datasets
umap_data = compute_all_embeddings_3d()

# Choose the dataset to visualize
TASK_DATASET_NAME = "fractal20220817_data"  # <-- change here

task_ind_output = widgets.Output()

def task_ind_on_change(change):
    with task_ind_output:
        clear_output(wait=True)
        points = get_dataset_points_3d(TASK_DATASET_NAME, umap_data)
        if points is None:
            print(f"❌ {TASK_DATASET_NAME} not found in UMAP cache")
            return
        fig = build_task_figure(points, TASK_DATASET_NAME)
        display(fig)

task_ind_on_change(None)
display(task_ind_output)

### Combined visualization

In [ ]:
task_dataset_names = sorted(set(umap_data["dataset"]))
task_checkboxes = {
    name: widgets.Checkbox(
        value=(i < 3),  # True for the first 3
        description=name,
        layout=widgets.Layout(width='350px')
    )
    for i, name in enumerate(task_dataset_names)
}

task_output = widgets.Output()
task_color_map = {name: COLORS[i % len(COLORS)] for i, name in enumerate(task_dataset_names)}

def task_on_change(change):
    with task_output:
        clear_output(wait=True)
        selected = [name for name, cb in task_checkboxes.items() if cb.value]
        if not selected:
            print("No datasets selected.")
            return

        fig = go.Figure()

        for dataset_name in selected:
            points = get_dataset_points_3d(dataset_name, umap_data)
            if points is None:
                print(f"  ⚠️ {dataset_name}: not in UMAP cache")
                continue

            robot, tag = get_dataset_label(dataset_name)
            n = len(points["x"])
            label = f"{dataset_name}<br>{robot} | {tag} | n={n}"
            color = task_color_map[dataset_name]

            fig.add_trace(make_task_trace_3d(points, label, color))

        fig.update_layout(
            title="Task Description Embedding — Combined 3D (UMAP)",
            scene=dict(
                xaxis_title='UMAP-1',
                yaxis_title='UMAP-2',
                zaxis_title='UMAP-3',
                aspectmode='cube'
            ),
            width=1000, height=800,
            legend=dict(
                x=1.05,
                y=0.5,
                itemsizing='constant',
                itemwidth=60
            )
        )
        display(fig)

for cb in task_checkboxes.values():
    cb.observe(task_on_change, names='value')

task_controls = widgets.VBox([
    widgets.HTML("<b>Datasets</b>"),
    widgets.VBox(list(task_checkboxes.values()))
])

task_on_change(None)
display(widgets.HBox([task_controls, task_output]))